# SortingHat baseline (self-contained)

Re-trains SortingHat's Random Forest (Shah et al., SIGMOD 2021) on the authors' released training data, checks it
against their held-out accuracy (paper: 0.9265), and applies it to our 76 test targets.

**Everything is inside this notebook:** the code, the authors' training data and our 76 test targets. You don't need a
Google Drive folder or any uploads.

**How to run**
1. Open this notebook in Colab (File › Upload notebook, or double-click it in Drive). A CPU runtime is enough.
2. Runtime › **Run all**. It takes about 10–20 min; step 3 prints one line per grid setting while it works.
3. `sortinghat_results.zip` downloads at the end. If the browser blocks the download, open the Files panel (folder icon
   on the left), right-click `sortinghat_results.zip` › Download. Put the zip in the project folder.

If the runtime disconnects or restarts, use Runtime › Run all again. Every step is safe to repeat.

## 1. Unpack the bundle

The first cell holds the embedded code and data (hidden); the second unpacks them.

In [ ]:
#@title Embedded bundle: code and data, ~959 kB (hidden) { display-mode: "form" }
PAYLOAD_SHA256 = "d7e8ea258a47bfd1310d7c0447e8ff2b9262ea3eeefb40e452e72d9810ef91e6"
PAYLOAD = """
...
"""

In [ ]:
import base64, hashlib, io, tarfile
from pathlib import Path

WORK = Path("/content/sortinghat") if Path("/content").is_dir() else Path.cwd() / "sortinghat_run"
raw = base64.b64decode("".join(PAYLOAD.split()))
assert hashlib.sha256(raw).hexdigest() == PAYLOAD_SHA256, "the embedded bundle is damaged: download the notebook again"
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(raw), mode="r:xz") as tar:
    if hasattr(tarfile, "data_filter"):
        tar.extractall(WORK, filter="data")
    else:
        tar.extractall(WORK)
print(f"unpacked to {WORK}: {len(list((WORK / 'store' / 'targets').glob('*.parquet')))} test targets")

## 2. Check the runtime

In [ ]:
import importlib, subprocess, sys
assert "WORK" in globals(), "run the cells from the top: Runtime › Run all"
try:
    import nltk  # preinstalled on Colab
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nltk"])
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
import sortinghat_lib as sh
importlib.reload(sh)
sh._nltk()  # downloads the NLTK stopwords and tokenizer data, so a network problem shows up here
print("ok", sh.versions())

## 3. Train, check, predict (about 10–20 min)

The last lines show the held-out accuracy next to the paper's 0.9265 and the feature types found for our targets.

In [ ]:
assert "sh" in globals(), "run the cells from the top: Runtime › Run all"
pred, types, report = sh.run(WORK, WORK / "store", WORK / "results")
print({k: v for k, v in report.items() if k != "grid"})

## 4. Download the results

In [ ]:
import shutil
assert (WORK / "results" / "predictions.csv").exists(), "step 3 has not finished: run all cells from the top"
zip_path = shutil.make_archive(str(WORK.parent / "sortinghat_results"), "zip", WORK / "results")
print("wrote", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass  # not on Colab: the zip is at the path above
except Exception as e:  # noqa: BLE001
    print(f"automatic download failed ({e}). Files panel › right-click sortinghat_results.zip › Download")